# Brain Activity Patterns Across Meditation Experience Levels
Neurotech@USC · Offline EEG preprocessing and feature extraction

Run from the project folder after installing the dependencies in the README. Add input filenames and pseudonymous participant and session IDs to the metadata CSV. The default `signal_only` mode reports observed trigger codes and creates unlabeled five-second windows. Select `meditation` only after checking a genuine timing source and its provenance.

The reusable modules keep the team's original working sequence of loading, reviewing, bandpass filtering, epoching, and QC. This notebook runs the same code as the command line.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from meditation_eeg.loading import discover
from meditation_eeg.pipeline import read_config, run

settings = read_config(Path("config.toml"))
files = (discover(settings["paths"]["data_dir"], settings["channels"]["legacy_mat_enabled"])
         if settings["paths"]["data_dir"].is_dir() else [])
print("Mode:", settings["mode"])
print("Inputs:", [path.name for path in files])
if not files:
    print("Add your recordings to the configured data folder and edit the metadata CSV.")

## Process recordings
Set `filename` to a single input filename to start with one session. Keep `interactive=False` for headless use. For manual review, use an interactive MNE browser, click channel names to mark bad channels, and press A to draw a `BAD_...` interval. Changes are recorded with that output run. Use a new output folder for reruns.

In [ ]:
filename = None
interactive = False
reports = run(settings, filename, interactive) if files else []

if reports:
    display(pd.DataFrame([{
        "participant": report["participant_id"],
        "session": report["session_id"],
        "trigger_codes": report["trigger_codes"],
        "candidate": report["candidate_epochs"],
        "retained": report["retained_epochs"],
        "rejected": report["rejected_epochs"],
        "reasons": dict(report["rejection_reasons"]),
    } for report in reports]))

## Inspect one session
Open each session's `qc.json` for acquisition metadata, stage provenance, annotation times, and component inspection details. Read `epoch_qc.csv` even when every window was rejected. The feature CSV will have headers and zero rows in that case.

In [ ]:
if reports:
    report = reports[0]
    folder = settings["paths"]["output_dir"] / report["participant_id"] / report["session_id"]
    display(pd.read_csv(folder / "channel_qc.csv"))
    display(pd.read_csv(folder / "features.csv").head())
    for name in ["eeg_samples.png", "psd_comparison.png", "power_over_time.png", "stage_power.png"]:
        if (folder / name).exists():
            display(Image(filename=str(folder / name)))